In [1]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd

df = pd.read_csv("../../data/interim/valeurs_manquantes_traitees.csv")
print(df.shape)

def cle_texte(texte):
    """2 libellés qui ne diffèrent que par la casse, les accents ou la ponctuation ont la même clé."""
    texte = str(texte).translate(str.maketrans({"’": "'", "‘": "'", "–": "-", "—": "-"}))
    texte = unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", " ", texte.lower()).strip()

(17498, 28)


# 2. Salaires bas

Certains diplômés à temps plein ont un salaire très bas. On veut savoir si ce sont des erreurs ou de vrais cas avant de décider quoi en faire.

On compare chaque salaire au SMIC net à temps plein de l'année où la personne a été interrogée. L'enquête a lieu en décembre, 2 ans après le diplôme pour les promos 2016 à 2020 (enquête à 30 mois) et 1 an après pour les promos 2022 à 2024 (enquête à 18 mois).

Le SMIC brut de chaque année vient de l'Urssaf et des Barèmes IPP (https://baremes.ipp.eu/parameters/marche-du-travail/marche_travail.salaire_minimum.smic/table/). Pour passer du brut au net, on utilise la fiche Smic de service-public.gouv.fr (https://www.service-public.gouv.fr/particuliers/vosdroits/F2300) : 1 867,02 € brut donnent 1 477,93 € net, donc on multiplie le brut par 0,7916.

In [2]:
# Source brut : Barèmes IPP / Urssaf, Smic mensuel 35 h en vigueur au 1er décembre de l'année d'enquête
# https://baremes.ipp.eu/parameters/marche-du-travail/marche_travail.salaire_minimum.smic/table/
smic_brut = {2018: 1498.47, 2019: 1521.22, 2020: 1539.42, 2021: 1589.47,
             2022: 1678.95, 2023: 1747.20, 2024: 1801.80, 2025: 1801.80}

# Coefficient net/brut : service-public.gouv.fr, fiche F2300 « Smic », tableau « Montants du Smic »
# Smic mensuel 2026 : 1 867,02 € brut -> 1 477,93 € net (déduction des cotisations salariales)
# 1 477,93 / 1 867,02 = 0,7916, appliqué aux Smic bruts des années précédentes

COEF_NET = 0.7916
smic_net = {annee: round(brut * COEF_NET) for annee, brut in smic_brut.items()}

annee_enquete = np.where(df["promotion"] <= 2020, df["promotion"] + 2, df["promotion"] + 1)
df["smic_net"] = pd.Series(annee_enquete, index=df.index).map(smic_net)

bas = df[df["revenu_net"] < df["smic_net"]].copy()
print(smic_net)
print(f"{len(bas)} lignes sous le SMIC net ({len(bas)/len(df):.2%})")
display(bas.groupby("promotion").size().to_frame("lignes sous le SMIC"))

{2018: 1186, 2019: 1204, 2020: 1219, 2021: 1258, 2022: 1329, 2023: 1383, 2024: 1426, 2025: 1426}
298 lignes sous le SMIC net (1.70%)


,lignes sous le SMIC
promotion,
2016,20
2017,48
2018,31
2019,40
2020,41
2022,40
2023,45
2024,33


In [3]:
# On regarde d'abord les métiers des personnes sous le SMIC
with pd.option_context("display.max_rows", None):
    display(bas["intitule_emploi"].value_counts())

intitule_emploi
Libraire                                                                              10
Vendeur·euse                                                                           5
Élève avocat-e                                                                         4
Avocat-e stagiaire                                                                     4
Doctorant-e                                                                            4
Traducteur·trice                                                                       4
Personne au pair                                                                       4
Chercheur-se doctorant-e                                                               4
Employé-e polyvalent-e                                                                 3
Préparateur·trice de commandes                                                         3
Doctorant·e                                                                            3
Pigis

En lisant cette liste, on voit revenir des statuts qui ne sont pas de vrais salaires : doctorants, élèves avocats, stagiaires, volontariat, au pair, indépendants. On en fait une liste de mots-clés pour les compter.

In [4]:
# Cas étonnants sous le SMIC on regarde où ils travaillent et dans quel secteur
etonnants = bas[bas["intitule_emploi"].str.contains("Auditeur|Psychologue|Chef.fe de projet", case=False, regex=True, na=False)]

colonnes = ["promotion", "intitule_emploi", "revenu_net", "smic_net", "lieu_emploi", "secteur_employeur", "categorie_emploi"]
with pd.option_context("display.max_rows", None, "display.width", 250):
    display(etonnants[colonnes].sort_values("intitule_emploi"))

print(pd.crosstab(etonnants["intitule_emploi"].str.extract(r"(Auditeur|Psychologue|Chef)", expand=False),
                  etonnants["lieu_emploi"].str.contains("tranger", na=False).map({True: "Étranger", False: "France"})))

,promotion,intitule_emploi,revenu_net,smic_net,lieu_emploi,secteur_employeur,categorie_emploi
13454,2017,Auditeur-trice financier-ère,1102.0,1204,Etranger,Privé,Cadre
13942,2017,Auditeur-trice financier-ère,1090.0,1204,Etranger,Privé,Cadre
13458,2017,Auditeur-trice sénior,991.0,1204,Etranger,Privé,Cadre
13475,2017,Auditeur-trice sénior,762.0,1204,Etranger,Privé,Cadre
13319,2017,Chef-fe de projet,1200.0,1204,Métropole Européenne de Lille,Associatif,Cadre
5663,2022,Chef·fe de projet,1300.0,1383,Métropole Européenne de Lille,Privé,Cadre
9293,2024,Chef·fe de projet,1400.0,1426,Hauts-de-France hors MEL,Privé,Cadre
3739,2020,Chef·fe de projet média,1250.0,1329,Île-de-France,Privé,Prof. interm.
8130,2023,Chef·fe de projet web,1400.0,1426,Hauts-de-France hors MEL,Associatif,Cadre
314,2019,Neuropsychologue,1188.0,1258,Reste de la France,Public,Cadre


lieu_emploi      France  Étranger
intitule_emploi                  
Auditeur              0         4
Chef                  5         0
Psychologue           2         0


Quelques métiers surprennent sous le SMIC : auditeurs, psychologues, chefs de projet. Les auditeurs travaillent tous à l'étranger, donc le SMIC français ne compte pas pour eux. Les psychologues et chefs de projet sont presque tous en France, payés entre 1 200 et 1 425 €, souvent 1 400 € tout rond. Ce sont de vrais petits salaires, pas des erreurs, donc on les garde aussi.

In [5]:
motifs = r"doctorant|avocat|au pair|volontari|VIE|stagiaire|pigiste|indépendant|auto-entrepreneur|assistant.{0,3} de langue|freelance|interne|élève"
bas["a_l_etranger"] = bas["lieu_emploi"].str.contains("tranger", na=False)
bas["statut_particulier"] = bas["intitule_emploi"].str.contains(motifs, case=False, regex=True, na=False)
display(pd.crosstab(bas["a_l_etranger"], bas["statut_particulier"], margins=True))

reste = bas[~bas["a_l_etranger"] & ~bas["statut_particulier"]]
print(f"Emplois en France sans statut particulier : {len(reste)}, salaire médian {reste['revenu_net'].median():.0f} €")
with pd.option_context("display.max_rows", 40, "display.max_colwidth", 60):
    display(reste[["intitule_emploi", "revenu_net", "mention"]].sort_values("revenu_net").head(40))

statut_particulier,False,True,All
a_l_etranger,,,
False,194,30,224
True,60,14,74
All,254,44,298


Emplois en France sans statut particulier : 194, salaire médian 1218 €


,intitule_emploi,revenu_net,mention
16988,Coach sportif-ve,450.0,Entraînement et optimisation de la performance sportive
15545,Micro-entrepreneur-se,578.0,"Traduction, interprétation"
15803,Clerc de notaire,600.0,Droit des affaires
9144,Coach sportif·ve,780.0,Staps - entraînement et optimisation de la performance s...
11550,Illustrateur-trice,800.0,Culture et communication
4046,Gérant·e pour une agence d'intérim et de recrutement,800.0,Management du sport
6708,Éducateur·trice de football et préparateur·trice mental·e,800.0,Staps - entraînement et optimisation de la performance s...
2333,Chargé·e de communication,800.0,"Information, communication"
16543,Web designer graphiste,800.0,"Marketing, vente"
13750,Juriste,900.0,Droit public


Conclusion : environ 1,7 % des lignes ont un salaire sous le SMIC. Ce ne sont pas des erreurs. On trouve trois cas :
- des gens qui travaillent à l'étranger, où le SMIC français ne compte pas ;
- des gens payés avec une indemnité plutôt qu'un vrai salaire (doctorants, élèves avocats, volontariat, au pair) ;
- des indépendants et des diplômés qui ont pris un emploi en dessous de leur niveau, souvent payés juste sous le SMIC.

Si on les supprimait, on enlèverait surtout des diplômés mal payés, alors que c'est justement ce qu'on veut prédire. Ils sont tous sous la médiane, donc ils ne faussent pas la cible. On les garde.

La colonne smic_net sert seulement pour cette vérification, on la supprime à la section 3. On ne crée pas de colonne "sous le SMIC", car elle utiliserait le salaire, c'est-à-dire la réponse qu'on cherche à prédire.

# 3. Recodages

Les PDF ne sont pas écrits pareil d'une année à l'autre. Une même modalité peut avoir plusieurs orthographes (apostrophe, virgules, majuscules, espace en trop). Les catégories changent aussi parfois. On montre d'abord tous ces problèmes, puis on les corrige en une seule fois.

#### 3.1 doublons de libellés

In [6]:
colonnes_texte = ["secteur_employeur", "lieu_emploi", "activite_employeur", "autre_diplome",
                  "categorie_emploi", "diplome", "mention", "parcours"]

for col in colonnes_texte:
    libelles = df[col].dropna().value_counts()
    cles = libelles.index.to_series().map(lambda t: cle_texte(t).replace(" ", ""))
    doublons = [libelles[cles == k].to_dict() for k in cles.unique() if (cles == k).sum() > 1]
    print(f"\n{col} : {len(libelles)} libellés, {len(doublons)} groupe(s) de doublons")
    for d in doublons[:5]:   # on affiche 5 exemples maximum
        print("   ", d)

# Intitulés de poste : même métier écrit avec "·" ou "-"
intitules = df["intitule_emploi"].dropna()
print("\nintitule_emploi :", intitules.nunique(), "intitulés,",
      intitules.str.replace("·", "-", regex=False).nunique(), "si on remplace · par -")
print(intitules[intitules.str.contains("Doctorant", na=False)].value_counts().head())


secteur_employeur : 4 libellés, 1 groupe(s) de doublons
    {'Associatif': 1992, 'Associati f': 4}

lieu_emploi : 7 libellés, 1 groupe(s) de doublons
    {'Étranger': 719, 'Etranger': 235}

activite_employeur : 34 libellés, 5 groupe(s) de doublons
    {'Comptabilité gestion conseil en management': 969, 'Comptabilité, gestion, conseil en management': 121}
    {'Ingénierie recherche et développement études techniques': 895, 'Ingénierie, recherche et développement, études techniques': 111}
    {"Activités financières et d'assurance": 786, 'Activités financières et d’assurance': 283}
    {'Édition presse communication': 224, 'Édition, presse, communication': 48}
    {'Agriculture,sylviculture et pêche': 20, 'Agriculture, sylviculture et pêche': 7}

autre_diplome : 24 libellés, 1 groupe(s) de doublons
    {'Autre Master': 896, 'Autre master': 163}

categorie_emploi : 3 libellés, 0 groupe(s) de doublons

diplome : 2 libellés, 0 groupe(s) de doublons

mention : 129 libellés, 17 groupe(s) de 

#### 3.2 : nomenclatures qui changent selon les années

In [7]:
with pd.option_context("display.max_rows", None, "display.width", 250):
    display(pd.crosstab(df["activite_employeur"], df["promotion"]))
    display(pd.crosstab(df["autre_diplome"], df["promotion"]))

promotion,2016,2017,2018,2019,2020,2022,2023,2024
activite_employeur,,,,,,,,
Activités de services administratifs et de soutien,50,75,48,62,69,67,69,58
Activités financières et d'assurance,122,114,141,146,115,148,0,0
Activités financières et d’assurance,0,22,0,0,0,0,129,132
Activités immobilières,33,26,27,28,25,22,28,21
"Activités informatiques, traitement, hébergement de données",127,191,186,219,141,210,180,130
Activités juridiques,34,32,34,43,38,25,24,12
"Activités spécialisées, scientifiques",0,0,0,10,0,0,0,0
"Activités spécialisées, scientifiques et techniques",0,60,0,0,0,0,0,0
Administration publique,150,217,242,273,264,243,274,259


promotion,2016,2017,2018,2019,2020,2022,2023,2024
autre_diplome,,,,,,,,
Aucun,1310,1729,1799,2175,1856,2136,2072,1807
Autre Master,93,147,158,44,150,158,146,0
Autre diplôme (bac+5),39,43,41,2,79,76,95,24
Autre diplôme (bac+6 ou plus),0,0,0,0,0,0,0,1
Autre diplôme (bac+6),55,51,43,1,50,31,17,0
Autre master,0,0,0,0,0,0,0,163
DU (bac+5),0,0,0,0,0,0,0,3
Diplôme d’IEP (bac+5),0,0,0,0,0,0,0,7
Diplôme d’une autre école (bac+5),0,0,0,0,0,0,0,19


Pour activite_employeur, certains libellés n'existent qu'une seule année. "Ingénierie RD..." (2016), "Industries" et "Autres activités de service" (2017) sont juste d'autres façons d'écrire des catégories qui existent déjà, on les fusionne. "Information et communication", "Commerce, transports, hébergement et restauration" et "Activités spécialisées, scientifiques..." viennent d'un classement plus large utilisé en 2017 et 2019. On les laisse comme ça : il y en a peu, et elles seront mises dans "Autre" au moment de l'encodage.

Pour autre_diplome, la promo 2024 utilise des catégories beaucoup plus détaillées que les autres années. On regroupe tout en 5 catégories pour que toutes les années soient comparables.

#### 3.3 : colonnes inutiles

In [8]:
print("Colonnes constantes :", [c for c in df.columns if df[c].nunique(dropna=False) <= 1])
print("type_emploi identique à emploi_stable :",
      f"{(df['type_emploi'].map({'Stable': True, 'Non stable': False}) == df['emploi_stable']).mean():.2%}")
display(df.groupby("promotion")["situation"].unique())

Colonnes constantes : ['effectif_insuffisant', 'temps_partiel', 'revenu_manquant']
type_emploi identique à emploi_stable : 99.99%


promotion
2016    [1er décembre 2018]
2017    [1er décembre 2019]
2018    [1er décembre 2020]
2019    [1er décembre 2021]
2020    [1er décembre 2022]
2022        [décembre 2023]
2023             [décembre]
2024             [décembre]
Name: situation, dtype: object

 #### 3.4  correction de tout en une fois de la section 2 et 3

In [9]:
def harmoniser(texte):
    if pd.isna(texte):
        return texte
    texte = texte.replace("’", "'")                # apostrophe ’ -> '
    texte = re.sub(r"\s*,\s*", ", ", texte)         # virgule toujours suivie d'un seul espace
    return re.sub(r"\s+", " ", texte).strip()      # espaces multiples -> un seul, et rien au début ni à la fin

for col in ["secteur_employeur", "lieu_emploi", "activite_employeur", "autre_diplome"]:
    df[col] = df[col].map(harmoniser)

df["secteur_employeur"] = df["secteur_employeur"].replace({"Associati f": "Associatif"})
df["lieu_emploi"] = df["lieu_emploi"].replace({"Etranger": "Étranger"})

df["activite_employeur"] = df["activite_employeur"].replace({
    "Comptabilité gestion conseil en management": "Comptabilité, gestion, conseil en management",
    "Ingénierie recherche et développement études techniques": "Ingénierie, recherche et développement, études techniques",
    "Ingénierie RD études techniques": "Ingénierie, recherche et développement, études techniques",
    "Édition presse communication": "Édition, presse, communication",
    "Industries": "Industrie",
    "Autres activités de service": "Autres activités de services",
    "Activités spécialisées, scientifiques": "Activités spécialisées, scientifiques et techniques",
})

def regrouper_diplome(texte):
    if pd.isna(texte):
        return texte
    t = texte.lower()
    if t == "aucun":
        return "Aucun"
    if "master" in t and "mastère" not in t:
        return "Autre master"
    if "ingénieur" in t:
        return "École d'ingénieurs"
    if "commerce" in t or "iep" in t or "sc po" in t or "mastère" in t:
        return "École de commerce, IEP ou mastère"
    return "Autre diplôme bac+5 ou plus"

df["autre_diplome"] = df["autre_diplome"].map(regrouper_diplome)

df["annee_enquete"] = np.where(df["promotion"] <= 2020, df["promotion"] + 2, df["promotion"] + 1)
df["delai_enquete_mois"] = np.where(df["promotion"] <= 2020, 30, 18)

a_supprimer = ["effectif_insuffisant", "temps_partiel", "type_emploi", "situation", "revenu_manquant", "smic_net"]
df = df.drop(columns=[c for c in a_supprimer if c in df.columns])

In [10]:
# Même mention ou parcours écrit de plusieurs façons : on garde l'écriture la plus fréquente
def cle_libelle(texte):
    texte = unicodedata.normalize("NFKD", str(texte).replace("’", "'")).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", texte.lower())   # sans accents, majuscules, ponctuation ni espaces

for col in ["mention", "parcours"]:
    avant = df[col].nunique()
    cles = df[col].map(cle_libelle, na_action="ignore")
    ecriture_la_plus_frequente = df.groupby(cles)[col].agg(lambda s: s.value_counts().idxmax())
    df[col] = cles.map(ecriture_la_plus_frequente)
    print(f"{col} : {avant} -> {df[col].nunique()} modalités")

# Intitulés : "Doctorant·e" et "Doctorant-e" deviennent identiques
avant = df["intitule_emploi"].nunique()
df["intitule_emploi"] = df["intitule_emploi"].str.replace("·", "-", regex=False).map(harmoniser)
print(f"intitule_emploi : {avant} -> {df['intitule_emploi'].nunique()} intitulés")

# Mentions 2016 non retrouvées : modalité à part plutôt que suppression
df["mention"] = df["mention"].fillna("Inconnue")

# Missions vides : texte vide (utile seulement si on fait du TF-IDF)
df["missions"] = df["missions"].fillna("")

mention : 129 -> 112 modalités
parcours : 553 -> 453 modalités
intitule_emploi : 8777 -> 8079 intitulés


Les mentions et les parcours ont le même problème que les activités : une même formation est écrite de plusieurs façons selon les années (majuscules, accents, tirets ou virgules). Par exemple "Comptabilité, contrôle, audit" et "Comptabilité - contrôle - audit". On les ramène à l'écriture la plus fréquente. Pour les intitulés de poste, "Doctorant·e" et "Doctorant-e" deviennent identiques. Les 416 mentions 2016 qu'on n'a pas pu retrouver deviennent "Inconnue" au lieu d'être supprimées.

### 3.5 vérification

In [ ]:
for col in ["secteur_employeur", "lieu_emploi", "activite_employeur", "autre_diplome"]:
    print(f"\n{col} : {df[col].nunique()} modalités")
    print(df[col].value_counts(dropna=False).to_string())

display(df.groupby("promotion")[["annee_enquete", "delai_enquete_mois"]].first())
print(df.shape)


secteur_employeur : 3 modalités
secteur_employeur
Privé         11045
Public         4457
Associatif     1996

lieu_emploi : 6 modalités
lieu_emploi
Métropole Européenne de Lille    6838
Île-de-France                    3316
Reste de la France               3173
Hauts-de-France hors MEL         3080
Étranger                          954
Nord sans précision               137

activite_employeur : 25 modalités
activite_employeur
Santé humaine et action sociale                                2537
Commerce                                                       2057
Administration publique                                        1922
Enseignement                                                   1552
Activités informatiques, traitement, hébergement de données    1384
Industrie                                                      1204
Ingénierie, recherche et développement, études techniques      1098
Comptabilité, gestion, conseil en management                   1090
Activités financières et

,annee_enquete,delai_enquete_mois
promotion,,
2016,2018,30
2017,2019,30
2018,2020,30
2019,2021,30
2020,2022,30
2022,2023,18
2023,2024,18
2024,2025,18


(17498, 25)


# 4. Variable cible

On veut savoir si un diplômé gagne plus que le salaire médian en France l'année où il a été interrogé. La médiane, c'est le salaire qui coupe les salariés en deux : la moitié gagne moins, l'autre moitié gagne plus.

On prend la médiane du salaire net en équivalent temps plein dans le secteur privé publiée par l'Insee (série 010752342, https://www.insee.fr/fr/statistiques/serie/010752342). L'Insee la donne par an, on la divise par 12 pour avoir un montant par mois.

On prend la médiane de l'année d'enquête (pas de l'année du diplôme), parce que c'est à ce moment-là que le salaire a été déclaré. L'Insee n'a pas encore publié 2025, donc pour la promo 2024 (interrogée en 2025) on prend la médiane 2024.

On utilise la même médiane pour tout le monde, même ceux qui travaillent dans le public ou l'associatif. La question devient : "gagne-t-il plus que le salarié du privé médian ?". Le secteur reste une variable du modèle.

La cible vaut 1 si le salaire est au-dessus de la médiane, 0 sinon.

In [ ]:
mediane_annuelle = {2018: 22201, 2019: 23009, 2020: 23683, 2021: 23667,
                    2022: 24514, 2023: 25568, 2024: 26279}
mediane_mensuelle = {annee: round(valeur / 12) for annee, valeur in mediane_annuelle.items()}

# 2025 pas encore publiée par l'Insee : on prend la dernière connue (2024)
mediane_mensuelle[2025] = mediane_mensuelle[2024]
print(mediane_mensuelle)

df["mediane_insee"] = df["annee_enquete"].map(mediane_mensuelle)
df["salaire_sup_mediane"] = (df["revenu_net"] > df["mediane_insee"]).astype(int)

print("Lignes sans médiane :", df["mediane_insee"].isna().sum())
print(f"Part au-dessus de la médiane : {df['salaire_sup_mediane'].mean():.1%}")
display(df.groupby("annee_enquete")["salaire_sup_mediane"].agg(["mean", "size"]).round(3))

{2018: 1850, 2019: 1917, 2020: 1974, 2021: 1972, 2022: 2043, 2023: 2131, 2024: 2190, 2025: 2190}
Lignes sans médiane : 0
Part au-dessus de la médiane : 58.5%


,mean,size
annee_enquete,,
2018,0.576,1579
2019,0.603,2103
2020,0.595,2161
2021,0.586,2252
2022,0.601,2212
2023,0.590,2548
2024,0.562,2443
2025,0.571,2200


Utilisation d'un LLM dans ce notebook

On s'est aidé d'un LLM (Claude) pour une partie du travail. Voici comment.

Ce qu'on lui a fait faire :
- recopier des tableaux en code pour éviter les erreurs de saisie à la main : les montants du SMIC brut par année, les médianes de salaire Insee, les correspondances entre anciens et nouveaux libellés (activités, autres diplômes) ;
- écrire les lignes de code qui demandent des expressions régulières (regex) ou un nettoyage de texte poussé ;

Comment on l'a guidé :
- on lui a donné les noms exacts des colonnes, leur type (texte, nombre, vrai/faux) et des exemples de valeurs réelles, pour que le code colle à notre jeu de données ;
- on lui a dit ce qu'on voulait obtenir (par exemple : une seule écriture par mention) plutôt que de le laisser choisir.

Ce qu'on a vérifié nous-mêmes :
- les chiffres recopiés ont été comparés aux sources officielles (Urssaf, service-public.gouv.fr, Insee) ;
- chaque cellule a été lancée et sa sortie relue (nombre de lignes, nombre de modalités avant et après) ;
- le LLM a parfois avancé des choses sans les vérifier (par exemple des montants du SMIC donnés de mémoire, ou l'idée que certains métiers mal payés étaient tous à l'étranger). On les a contrôlées dans les données avant de les garder.

Les décisions restent les nôtres : garder ou retirer des lignes (salaires bas, emplois à l'étranger), choisir la médiane de référence, choisir les regroupements.